# 02 — Analytical Decisions

## Project: Evaluating Low-Cost PM2.5 Sensors: Calibration Accuracy, Generalization, and Reliability

### 1. Purpose of This Notebook

This notebook documents the analytical decisions and methodological considerations that will guide the exploratory data analysis, calibration modeling, and evaluation stages of this project.

The previous notebook (`01_data_profile.ipynb`) focused on understanding the structure, completeness, consistency, and quality of the two source datasets.

The purpose of this notebook is to translate the findings from the data-profile stage and supporting research into clear, justified, and reproducible analytical principles.

Not every analytical decision can be finalized before exploratory data analysis. Therefore, this notebook distinguishes between:
- **Established decisions:** Approaches supported by the available dataset evidence, documentation, or published research.
- **Provisional decisions:** Initial approaches that require further examination or empirical evaluation.
- **Open questions:** Issues that cannot be resolved until additional analysis or clarification is available.

### 2. Research Question

How much does calibration improve agreement between low-cost PurpleAir sensors and reference monitors, and does the improvement hold under wildfire-smoke conditions?

### 3. Datasets

This project uses two datasets:

- **Dataset A — 24-Hour Collocation:** Contains daily PurpleAir measurements and reference-monitor measurements (FEM/FRM), along with environmental variables and existing correction fields.
- **Dataset B — Hourly Wildfire Smoke:** Contains hourly raw PurpleAir measurements, reference-monitor measurements, relative humidity, and previously published adjusted measurements under different correction and withholding configurations.

The datasets have different temporal resolutions and sampling structures. Their roles and limitations will therefore be considered separately during analysis.

### 4. Analytical Decisions

The following decisions and methodological questions are documented in this notebook:

1. Reference-monitor selection and handling of multiple FEM/FRM readings.
2. Treatment of nonpositive reference measurements.
3. Handling of missing meteorological variables.
4. Selection of the raw PurpleAir PM2.5 measurement.
5. Selection of meteorological predictors for calibration.
6. Treatment of repeated observations and prevention of data leakage.
7. Selection of model evaluation metrics.
8. Calibration model comparison.
9. Evaluation of generalization under wildfire-smoke conditions.
10. Data preprocessing and transformation principles.

Each decision includes relevant evidence, the analytical reasoning, the selected or proposed approach, and any limitations or sensitivity analyses that may be necessary.

### 5. Workflow

The analysis will follow this sequence:

1. Review relevant findings from the data-profile notebook and supporting documentation.
2. Evaluate analytical options using the available evidence.
3. Establish decisions that can be justified before EDA.
4. Document provisional approaches and unresolved questions.
5. Conduct exploratory data analysis to examine distributions, relationships, temporal patterns, regional differences, and potential data-quality concerns.
6. Revisit and finalize decisions that depend on EDA findings.
7. Proceed to calibration modeling and evaluation using the finalized analytical framework.

This approach avoids making data-dependent assumptions before the relevant patterns and limitations have been examined.

### 6. Expected Outcome

By the end of this notebook, the project will have a documented initial analytical framework covering reference selection, data handling, model inputs, evaluation metrics, and validation considerations.

Some decisions will remain provisional until EDA provides sufficient evidence. Any changes to the initial analytical framework will be documented and justified.

No calibration models will be trained in this notebook. Model development and performance evaluation will be conducted in subsequent notebooks.

---

**Note:** All original source data will remain unchanged. Any filtering, transformations, derived variables, or analytical subsets will be created separately and documented to maintain reproducibility.

## Decision 1 — Reference Monitor Selection in Dataset A

### Analytical Question

Dataset A contains PM2.5 reference measurements collected using Federal Equivalent Methods (FEM) and Federal Reference Methods (FRM).

Some PurpleAir site-date groups contain only FEM measurements, some contain only FRM measurements, and others contain both. Some groups also contain multiple readings within the same reference category.

In this section, we will investigate:

1. Whether FEM and FRM measurements should be combined or analyzed separately.
2. Whether one reference category should be prioritized when both are available.
3. How multiple reference readings associated with the same PurpleAir site-date group should be handled.

We will examine the dataset structure, relevant published research, and corresponding EPA AQS records before establishing the final analytical approach.

**The final decision will be documented after the supporting analysis and findings.**

In [1]:

from pathlib import Path
import pandas as pd

# Find the project root from the notebook's current directory.
current_dir = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in (current_dir, *current_dir.parents)
        if (path / "data" / "raw").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root. Open this notebook "
        "from inside the low-cost-pm25-calibration repository."
    )

RAW_DIR = PROJECT_ROOT / "data" / "raw"

COLLOCATION_DIR = RAW_DIR / "24hr_collocation"
SMOKE_DIR = RAW_DIR / "hourly_smoke"

COLLOCATION_FILE = COLLOCATION_DIR / "Full24hrdataset.csv"
SMOKE_FILE = (
    SMOKE_DIR
    / "Finaldataset_correcteddatasetwholding_10_26_22.csv"
)

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DIR)


Project root: .
Raw data directory: data/raw


In [2]:

# Verify the main CSV files before attempting to load them.
required_files = {
    "24-hour collocation dataset": COLLOCATION_FILE,
    "Hourly wildfire-smoke dataset": SMOKE_FILE,
}

for dataset_name, file_path in required_files.items():
    if not file_path.is_file():
        raise FileNotFoundError(
            f"{dataset_name} is missing: {file_path}\n"
            "Run `python src/download_data.py` from the project root."
        )

    print(
        f"{dataset_name}: available "
        f"({file_path.stat().st_size:,} bytes)"
    )


24-hour collocation dataset: available (6,420,524 bytes)
Hourly wildfire-smoke dataset: available (261,969,901 bytes)


In [3]:
# Load the main 24-hour CSV without modifying the source file.
df_24hr = pd.read_csv(COLLOCATION_FILE, low_memory=False)



In [4]:

# Check the number of FEM and FRM readings within each ID-Date-region group

reference_counts = (
    df_24hr
    .groupby(["ID", "Date", "region", "FEMorFRM"], dropna=False)
    .size()
    .unstack(fill_value=0)
)

for method in ["FEM", "FRM"]:
    if method not in reference_counts.columns:
        reference_counts[method] = 0

paired_groups = reference_counts[
    (reference_counts["FEM"] > 0) &
    (reference_counts["FRM"] > 0)
]

print("Paired groups:", len(paired_groups))

print("\nFEM readings per paired group:")
print(paired_groups["FEM"].value_counts().sort_index())

print("\nFRM readings per paired group:")
print(paired_groups["FRM"].value_counts().sort_index())

print("\nGroups with exactly one FEM and one FRM:")
print(
    (
        (paired_groups["FEM"] == 1) &
        (paired_groups["FRM"] == 1)
    ).sum()
)


Paired groups: 1146

FEM readings per paired group:
FEM
1    731
2    415
Name: count, dtype: int64

FRM readings per paired group:
FRM
1    1037
2     109
Name: count, dtype: int64

Groups with exactly one FEM and one FRM:
665


In [5]:

# Cross-tabulate FEM and FRM reading counts in paired groups

print("FEM × FRM reading-count combinations:")
print(
    pd.crosstab(
        paired_groups["FEM"],
        paired_groups["FRM"]
    )
)

print("\nGroups with multiple readings of both types:")
print(
    (
        (paired_groups["FEM"] > 1) &
        (paired_groups["FRM"] > 1)
    ).sum()
)


FEM × FRM reading-count combinations:
FRM    1   2
FEM         
1    665  66
2    372  43

Groups with multiple readings of both types:
43


In [6]:

# Examine reference-monitor type combinations in paired groups

paired_keys = paired_groups.index

paired_records = (
    df_24hr
    .set_index(["ID", "Date", "region"])
    .loc[paired_keys]
    .reset_index()
)

print("Reference monitor types in paired groups:")
print(
    paired_records.groupby(
        ["FEMorFRM", "FEMorFRMtype"],
        dropna=False
    ).size().to_string()
)

print("\nMonitor types within groups containing multiple readings:")
multiple_keys = paired_groups.index[
    (paired_groups["FEM"] > 1) |
    (paired_groups["FRM"] > 1)
]

multiple_records = (
    df_24hr
    .set_index(["ID", "Date", "region"])
    .loc[multiple_keys]
    .reset_index()
)

print(
    multiple_records.groupby(
        ["ID", "Date", "region", "FEMorFRM", "FEMorFRMtype"],
        dropna=False
    ).size().to_string()
)


Reference monitor types in paired groups:
FEMorFRM  FEMorFRMtype                                                         
FEM       Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation                   357
          Met One BAM-1022 Mass Monitor w/ VSCC or TE-PM2.5C - Beta Attenuation     328
          Teledyne T640 at 5.0 LPM - Broadband spectroscopy                         607
          Teledyne T640X at 16.67 LPM - Broadband spectroscopy                      236
          Thermo Scientific TEOM 1405-DF Dichotomous FDMS - FDMS Gravimetric         33
FRM       R & P Model 2025 PM-2.5 Sequential Air Sampler w/VSCC - Gravimetric      1255

Monitor types within groups containing multiple readings:
ID    Date        region     FEMorFRM  FEMorFRMtype                                                         
GA1   2019-08-07  Southeast  FEM       Teledyne T640X at 16.67 LPM - Broadband spectroscopy                     1
                             FRM       R & P Model 2025 PM-2.5 Seque

In [7]:

# Count distinct reference-monitor types within each ID-Date-region group

monitor_type_counts = (
    df_24hr
    .groupby(
        ["ID", "Date", "region", "FEMorFRM"],
        dropna=False
    )
    .agg(
        number_of_readings=("FEMorFRMtype", "size"),
        distinct_monitor_types=("FEMorFRMtype", "nunique")
    )
    .reset_index()
)

# Focus on groups with multiple readings of the same reference category
multiple_reading_types = monitor_type_counts[
    monitor_type_counts["number_of_readings"] > 1
]

print("Multiple-reading groups by reference category:")
print(
    multiple_reading_types["FEMorFRM"]
    .value_counts()
)

print("\nDistinct monitor types within multiple-reading groups:")
print(
    pd.crosstab(
        multiple_reading_types["FEMorFRM"],
        multiple_reading_types["distinct_monitor_types"]
    )
)

print("\nGroups with multiple readings and multiple monitor types:")
print(
    multiple_reading_types[
        multiple_reading_types["distinct_monitor_types"] > 1
    ].to_string(index=False)
)


Multiple-reading groups by reference category:
FEMorFRM
FEM    1357
FRM     220
Name: count, dtype: int64

Distinct monitor types within multiple-reading groups:
distinct_monitor_types     1   2
FEMorFRM                        
FEM                     1316  41
FRM                      137  83

Groups with multiple readings and multiple monitor types:
 ID       Date region FEMorFRM  number_of_readings  distinct_monitor_types
AK1 2019-05-15 Alaska      FRM                   2                       2
AK1 2019-05-18 Alaska      FRM                   2                       2
AK1 2019-05-21 Alaska      FRM                   2                       2
AK1 2019-05-24 Alaska      FRM                   2                       2
AK1 2019-05-30 Alaska      FRM                   2                       2
AK1 2019-06-02 Alaska      FRM                   2                       2
AK1 2019-06-05 Alaska      FRM                   2                       2
AK1 2019-06-08 Alaska      FRM                 

In [8]:

# Examine repeated readings specifically within FEM-FRM paired groups

group_details = (
    df_24hr
    .groupby(
        ["ID", "Date", "region", "FEMorFRM"],
        dropna=False
    )
    .agg(
        number_of_readings=("PM25FM", "size"),
        distinct_monitor_types=("FEMorFRMtype", "nunique"),
        distinct_reference_values=("PM25FM", "nunique")
    )
)

# Keep only groups that contain both FEM and FRM
paired_details = group_details.loc[
    group_details.index.droplevel("FEMorFRM").isin(paired_groups.index)
].reset_index()

print("Paired-group repeated readings:")
repeated_paired = paired_details[
    paired_details["number_of_readings"] > 1
]

print(
    pd.crosstab(
        repeated_paired["FEMorFRM"],
        [
            repeated_paired["distinct_monitor_types"],
            repeated_paired["distinct_reference_values"]
        ]
    )
)

print("\nRepeated paired groups with different reference values:")
print(
    repeated_paired[
        repeated_paired["distinct_reference_values"] > 1
    ].groupby("FEMorFRM").size()
)

print("\nRepeated paired groups with identical reference values:")
print(
    repeated_paired[
        repeated_paired["distinct_reference_values"] == 1
    ].groupby("FEMorFRM").size()
)


Paired-group repeated readings:
distinct_monitor_types      1        2
distinct_reference_values   1    2   2
FEMorFRM                              
FEM                         4  390  21
FRM                        19   90   0

Repeated paired groups with different reference values:
FEMorFRM
FEM    411
FRM     90
dtype: int64

Repeated paired groups with identical reference values:
FEMorFRM
FEM     4
FRM    19
dtype: int64


In [10]:
# Count FEM and FRM readings within each ID-Date-region group

reading_counts = (
    group_details["number_of_readings"]
    .unstack("FEMorFRM")
    .rename(columns={"FEM": "FEM_readings", "FRM": "FRM_readings"})
    .reset_index()
)

# Keep only ID-Date-region groups containing both FEM and FRM
paired_reading_counts = reading_counts[
    reading_counts["FEM_readings"].notna() &
    reading_counts["FRM_readings"].notna()
].copy()

# Identify groups with exactly one reading of each type
one_to_one_groups = paired_reading_counts[
    (paired_reading_counts["FEM_readings"] == 1) &
    (paired_reading_counts["FRM_readings"] == 1)
]

print("Total paired groups:", len(paired_reading_counts))
print("One-to-one paired groups:", len(one_to_one_groups))
print(
    "Percentage retained:",
    round(len(one_to_one_groups) / len(paired_reading_counts) * 100, 2),
    "%"
)

print(
    "Paired groups excluded by the one-to-one rule:",
    len(paired_reading_counts) - len(one_to_one_groups)
)

Total paired groups: 1146
One-to-one paired groups: 665
Percentage retained: 58.03 %
Paired groups excluded by the one-to-one rule: 481


In [11]:
# Compare one-to-one and repeated paired groups by region

paired_reading_counts["group_type"] = (
    (paired_reading_counts["FEM_readings"] == 1) &
    (paired_reading_counts["FRM_readings"] == 1)
).map({
    True: "One-to-one",
    False: "Repeated"
})

region_comparison = pd.crosstab(
    paired_reading_counts["region"],
    paired_reading_counts["group_type"],
    margins=True
)

print(region_comparison)

print("\nPercentage within each region:")
print(
    pd.crosstab(
        paired_reading_counts["region"],
        paired_reading_counts["group_type"],
        normalize="index"
    ).mul(100).round(2)
)

group_type         One-to-one  Repeated   All
region                                       
Central Southwest          78         0    78
North                     406       435   841
Southeast                 181        46   227
All                       665       481  1146

Percentage within each region:
group_type         One-to-one  Repeated
region                                 
Central Southwest      100.00      0.00
North                   48.28     51.72
Southeast               79.74     20.26


In [12]:
# Compare one-to-one and repeated paired groups by region
# and reference-monitor category

region_monitor_comparison = pd.crosstab(
    [
        paired_reading_counts["region"],
        paired_reading_counts["FEM_readings"].gt(1).map({
            True: "Multiple FEM",
            False: "One FEM"
        })
    ],
    paired_reading_counts["FRM_readings"].gt(1).map({
        True: "Multiple FRM",
        False: "One FRM"
    })
)

print(region_monitor_comparison)

FRM_readings                    Multiple FRM  One FRM
region            FEM_readings                       
Central Southwest One FEM                  0       78
North             Multiple FEM            43      372
                  One FEM                 20      406
Southeast         One FEM                 46      181


In [13]:
# Inspect actual rows from repeated FEM/FRM groups
# that are also paired (both FEM and FRM are available).

paired_keys = set(paired_groups.index.tolist())

repeated_details = (
    group_details
    .reset_index()
    .query("number_of_readings > 1")
    .copy()
)

repeated_details["sensor_date_region"] = list(
    zip(
        repeated_details["ID"],
        repeated_details["Date"],
        repeated_details["region"]
    )
)

repeated_paired_details = repeated_details[
    repeated_details["sensor_date_region"].isin(paired_keys)
].copy()

# Inspect groups with multiple readings but only one monitor type,
# and groups with multiple monitor types.
for category in ["FEM", "FRM"]:

    category_groups = repeated_paired_details[
        repeated_paired_details["FEMorFRM"] == category
    ]

    print(f"\n{'=' * 65}")
    print(f"{category}: EXAMPLE REPEATED GROUPS")
    print(f"{'=' * 65}")

    for type_count in [1, 2]:

        examples = category_groups[
            category_groups["distinct_monitor_types"] == type_count
        ].head(2)

        for _, group in examples.iterrows():

            print(
                f"\nID: {group['ID']} | "
                f"Date: {group['Date']} | "
                f"Region: {group['region']} | "
                f"Readings: {group['number_of_readings']} | "
                f"Monitor types: {type_count}"
            )

            rows = df_24hr[
                (df_24hr["ID"] == group["ID"]) &
                (df_24hr["Date"] == group["Date"]) &
                (df_24hr["region"] == group["region"]) &
                (df_24hr["FEMorFRM"] == category)
            ]

            display(
                rows[
                    [
                        "ID",
                        "Date",
                        "region",
                        "FEMorFRM",
                        "FEMorFRMtype",
                        "PM25FM",
                        "PM25cf1",
                        "PM25cfatm"
                    ]
                ]
            )


FEM: EXAMPLE REPEATED GROUPS

ID: IA1 | Date: 2018-05-13 | Region: North | Readings: 2 | Monitor types: 1


,ID,Date,region,FEMorFRM,FEMorFRMtype,PM25FM,PM25cf1,PM25cfatm
6941,IA1,2018-05-13,North,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,13.750000,33.047216,30.835477
7319,IA1,2018-05-13,North,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,11.958333,33.047216,30.835477



ID: IA1 | Date: 2018-07-15 | Region: North | Readings: 2 | Monitor types: 1


,ID,Date,region,FEMorFRM,FEMorFRMtype,PM25FM,PM25cf1,PM25cfatm
6532,IA1,2018-07-15,North,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,15.041667,27.633556,25.996485
6578,IA1,2018-07-15,North,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,17.333333,27.633556,25.996485



ID: IA1 | Date: 2019-09-20 | Region: North | Readings: 2 | Monitor types: 2


,ID,Date,region,FEMorFRM,FEMorFRMtype,PM25FM,PM25cf1,PM25cfatm
6733,IA1,2019-09-20,North,FEM,Teledyne T640 at 5.0 LPM - Broadband spectroscopy,15.258333,26.696944,26.116674
6818,IA1,2019-09-20,North,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta At...,14.750000,26.696944,26.116674


... (2 outputs left out)


FRM: EXAMPLE REPEATED GROUPS

ID: GA1 | Date: 2019-08-07 | Region: Southeast | Readings: 2 | Monitor types: 1


,ID,Date,region,FEMorFRM,FEMorFRMtype,PM25FM,PM25cf1,PM25cfatm
5349,GA1,2019-08-07,Southeast,FRM,R & P Model 2025 PM-2.5 Sequential Air Sampler...,11.2,22.340403,22.256868
5418,GA1,2019-08-07,Southeast,FRM,R & P Model 2025 PM-2.5 Sequential Air Sampler...,11.3,22.340403,22.256868



ID: GA1 | Date: 2019-08-10 | Region: Southeast | Readings: 2 | Monitor types: 1


,ID,Date,region,FEMorFRM,FEMorFRMtype,PM25FM,PM25cf1,PM25cfatm
5351,GA1,2019-08-10,Southeast,FRM,R & P Model 2025 PM-2.5 Sequential Air Sampler...,11.0,19.211027,19.210845
5419,GA1,2019-08-10,Southeast,FRM,R & P Model 2025 PM-2.5 Sequential Air Sampler...,11.5,19.211027,19.210845


In [14]:
# Check whether PurpleAir values are identical within repeated
# FEM/FRM groups in paired ID-Date-region combinations.

repeated_paired_keys = (
    repeated_paired_details[
        ["ID", "Date", "region", "FEMorFRM"]
    ]
)

repeated_rows = df_24hr.merge(
    repeated_paired_keys,
    on=["ID", "Date", "region", "FEMorFRM"],
    how="inner"
)

purpleair_consistency = (
    repeated_rows
    .groupby(["ID", "Date", "region", "FEMorFRM"])
    .agg(
        cf1_unique=("PM25cf1", "nunique"),
        cf_atm_unique=("PM25cfatm", "nunique"),
        reference_unique=("PM25FM", "nunique")
    )
)

print("Repeated paired groups:", len(purpleair_consistency))

print("\nPM25cf1 consistency:")
print(purpleair_consistency["cf1_unique"].value_counts().sort_index())

print("\nPM25cfatm consistency:")
print(purpleair_consistency["cf_atm_unique"].value_counts().sort_index())

print("\nReference value variation:")
print(purpleair_consistency["reference_unique"].value_counts().sort_index())

Repeated paired groups: 524

PM25cf1 consistency:
cf1_unique
1    524
Name: count, dtype: int64

PM25cfatm consistency:
cf_atm_unique
1    524
Name: count, dtype: int64

Reference value variation:
reference_unique
1     23
2    501
Name: count, dtype: int64


In [17]:
monitor_file = Path("../data/raw/aqs_monitors.csv")

print("Expected file location:", monitor_file.resolve())
print("File exists:", monitor_file.exists())

if not monitor_file.exists():
    raise FileNotFoundError(f"File not found: {monitor_file.resolve()}")

aqs_monitors = pd.read_csv(
    monitor_file,
    dtype=str,
    low_memory=False
)

print("Shape:", aqs_monitors.shape)
print("\nColumns:")
print(aqs_monitors.columns.tolist())

display(aqs_monitors.head())

Expected file location: data/raw/aqs_monitors.csv
File exists: True
Shape: (370090, 32)

Columns:
['State Code', 'County Code', 'Site Number', 'Parameter Code', 'Parameter Name', 'POC', 'Latitude', 'Longitude', 'Datum', 'First Year of Data', 'Last Sample Date', 'Monitor Type', 'Networks', 'Reporting Agency', 'PQAO', 'Collecting Agency', 'Exclusions', 'Monitoring Objective', 'Last Method Code', 'Last Method', 'Measurement Scale', 'Measurement Scale Definition', 'NAAQS Primary Monitor', 'QA Primary Monitor', 'Local Site Name', 'Address', 'State Name', 'County Name', 'City Name', 'CBSA Name', 'Tribe Name', 'Extraction Date']


,State Code,County Code,Site Number,Parameter Code,Parameter Name,POC,Latitude,Longitude,Datum,First Year of Data,...,NAAQS Primary Monitor,QA Primary Monitor,Local Site Name,Address,State Name,County Name,City Name,CBSA Name,Tribe Name,Extraction Date
0,01,001,0001,11103,Benzene soluble organics (TSP),1,32.437458,-86.472891,WGS84,1974,...,NaN,NaN,NaN,"KING ARTHUR TRAILER COURT, PRATTVILLE,AL",Alabama,Autauga,Prattville,"Montgomery, AL",NaN,2026-06-26
1,01,001,0001,42401,Sulfur dioxide,1,32.437458,-86.472891,WGS84,1974,...,NaN,NaN,NaN,"KING ARTHUR TRAILER COURT, PRATTVILLE,AL",Alabama,Autauga,Prattville,"Montgomery, AL",NaN,2026-06-26
2,01,001,0002,42401,Sulfur dioxide,1,32.42847,-86.443585,WGS84,1980,...,NaN,NaN,NaN,COUNTY RD 4 PRATTVILLE EXPERIMENT ST,Alabama,Autauga,Prattville,"Montgomery, AL",NaN,2026-06-26
3,01,001,0002,44201,Ozone,1,32.42847,-86.443585,WGS84,1980,...,NaN,NaN,NaN,COUNTY RD 4 PRATTVILLE EXPERIMENT ST,Alabama,Autauga,Prattville,"Montgomery, AL",NaN,2026-06-26
4,01,001,0003,44201,Ozone,1,32.332659,-86.791521,WGS84,1989,...,NaN,NaN,NaN,"1170 COUNTY RD.15 SO., SELMA, AL. 36701",Alabama,Autauga,Not in a City,"Montgomery, AL",NaN,2026-06-26


In [18]:
ia1_monitors = aqs_monitors[
    (aqs_monitors["State Code"] == "19") &
    (aqs_monitors["County Code"] == "113") &
    (aqs_monitors["Site Number"] == "0040") &
    (aqs_monitors["Parameter Code"] == "88101")
].copy()

print("IA1 PM2.5 monitor records:", len(ia1_monitors))

display(
    ia1_monitors[
        [
            "State Code",
            "County Code",
            "Site Number",
            "Parameter Code",
            "POC",
            "Parameter Name",
            "First Year of Data",
            "Last Sample Date",
            "Monitor Type",
            "Last Method Code",
            "Last Method",
            "Reporting Agency",
            "PQAO",
            "Local Site Name"
        ]
    ].sort_values("POC")
)

IA1 PM2.5 monitor records: 6


,State Code,County Code,Site Number,Parameter Code,POC,Parameter Name,First Year of Data,Last Sample Date,Monitor Type,Last Method Code,Last Method,Reporting Agency,PQAO,Local Site Name
117520,19,113,0040,88101,1,PM2.5 - Local Conditions,2008,2026-05-31,SLAMS,145,R & P Model 2025 PM-2.5 Sequential Air Sampler...,Linn County Health Department,Linn County Health Department,Public Health
117521,19,113,0040,88101,2,PM2.5 - Local Conditions,2013,2026-05-26,SLAMS,145,R & P Model 2025 PM-2.5 Sequential Air Sampler...,Linn County Health Department,Linn County Health Department,Public Health
117524,19,113,0040,88101,23,PM2.5 - Local Conditions,2019,2023-12-31,SPM,736,Teledyne T640 at 5.0 LPM (Corrected) - Broadba...,Linn County Health Department,Linn County Health Department,Public Health
117525,19,113,0040,88101,24,PM2.5 - Local Conditions,2023,2023-12-31,SPM,736,Teledyne T640 at 5.0 LPM (Corrected) - Broadba...,Linn County Health Department,Linn County Health Department,Public Health
117522,19,113,0040,88101,3,PM2.5 - Local Conditions,2016,2026-05-31,SPM,636,Teledyne T640 at 5.0 LPM w/Network Data Alignm...,Linn County Health Department,Linn County Health Department,Public Health
117523,19,113,0040,88101,4,PM2.5 - Local Conditions,2016,2026-05-31,SPM,636,Teledyne T640 at 5.0 LPM w/Network Data Alignm...,Linn County Health Department,Linn County Health Department,Public Health


In [19]:
ia1_repeated = df_24hr[
    (df_24hr["ID"] == "IA1") &
    (df_24hr["FEMorFRM"].isin(["FEM", "FRM"]))
].copy()

ia1_repeated_summary = (
    ia1_repeated
    .groupby(["Date", "FEMorFRM"])
    .agg(
        rows=("PM25FM", "size"),
        reference_values=("PM25FM", lambda x: sorted(x.dropna().unique())),
        monitor_types=("FEMorFRMtype", lambda x: sorted(x.dropna().unique())),
        purpleair_cf1=("PM25cf1", "first"),
        purpleair_cfatm=("PM25cfatm", "first")
    )
    .reset_index()
)

ia1_repeated_summary = ia1_repeated_summary[
    ia1_repeated_summary["rows"] > 1
].sort_values(["Date", "FEMorFRM"])

display(ia1_repeated_summary)

,Date,FEMorFRM,rows,reference_values,monitor_types,purpleair_cf1,purpleair_cfatm
3,2018-05-08,FRM,2,"[8.4, 8.6]",[R & P Model 2025 PM-2.5 Sequential Air Sample...,8.491267,8.338796
6,2018-05-12,FEM,2,"[5.083333, 7.666667]",[Met One BAM-1020 Mass Monitor w/VSCC - Beta A...,17.849707,17.844327
7,2018-05-13,FEM,2,"[11.958333, 13.75]",[Met One BAM-1020 Mass Monitor w/VSCC - Beta A...,33.047216,30.835477
10,2018-05-14,FRM,2,"[13.5, 13.7]",[R & P Model 2025 PM-2.5 Sequential Air Sample...,33.717493,30.938742
22,2018-05-26,FEM,2,"[5.166667, 7.166667]",[Met One BAM-1020 Mass Monitor w/VSCC - Beta A...,9.684387,9.168863
...,...,...,...,...,...,...,...
464,2019-12-13,FEM,2,"[27.304167, 27.833333]",[Met One BAM-1020 Mass Monitor w/VSCC - Beta A...,39.647590,38.159535
472,2019-12-20,FEM,2,"[13.983333, 14.875]",[Met One BAM-1020 Mass Monitor w/VSCC - Beta A...,20.284437,20.283535
475,2019-12-23,FRM,2,"[14.1, 15.1]",[R & P Model 2025 PM-2.5 Sequential Air Sample...,26.271875,25.882229
476,2019-12-24,FEM,2,"[16.791667, 17.683333]",[Met One BAM-1020 Mass Monitor w/VSCC - Beta A...,28.759285,28.342938


In [20]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

display(ia1_monitors.T)

,117520,117521,117522,117523,117524,117525
State Code,19,19,19,19,19,19
County Code,113,113,113,113,113,113
Site Number,0040,0040,0040,0040,0040,0040
Parameter Code,88101,88101,88101,88101,88101,88101
Parameter Name,PM2.5 - Local Conditions,PM2.5 - Local Conditions,PM2.5 - Local Conditions,PM2.5 - Local Conditions,PM2.5 - Local Conditions,PM2.5 - Local Conditions
POC,1,2,3,4,23,24
Latitude,41.97677,41.97677,41.97677,41.97677,41.97677,41.97677
Longitude,-91.68766,-91.68766,-91.68766,-91.68766,-91.68766,-91.68766
Datum,WGS84,WGS84,WGS84,WGS84,WGS84,WGS84
First Year of Data,2008,2013,2016,2016,2019,2023


In [21]:
import zipfile

raw_dir = Path("../data/raw")

zip_files = [
    "daily_88101_2018.zip",
    "daily_88101_2019.zip",
    "daily_88101_2020.zip"
]

for filename in zip_files:
    file_path = raw_dir / filename

    print(f"\n{filename}")
    print("Exists:", file_path.exists())

    if file_path.exists():
        print("Size (MB):", round(file_path.stat().st_size / (1024 ** 2), 2))

        with zipfile.ZipFile(file_path, "r") as archive:
            print("Contents:", archive.namelist())


daily_88101_2018.zip
Exists: True
Size (MB): 6.03
Contents: ['daily_88101_2018.csv']

daily_88101_2019.zip
Exists: True
Size (MB): 6.98
Contents: ['daily_88101_2019.csv']

daily_88101_2020.zip
Exists: True
Size (MB): 7.55
Contents: ['daily_88101_2020.csv']


In [22]:

file_path = raw_dir / "daily_88101_2018.zip"

with zipfile.ZipFile(file_path, "r") as archive:
    with archive.open("daily_88101_2018.csv") as csv_file:
        aqs_2018_sample = pd.read_csv(
            csv_file,
            dtype=str,
            nrows=5
        )

print("Columns:")
print(aqs_2018_sample.columns.tolist())

print("\nSample records:")
display(aqs_2018_sample.T)

Columns:
['State Code', 'County Code', 'Site Num', 'Parameter Code', 'POC', 'Latitude', 'Longitude', 'Datum', 'Parameter Name', 'Sample Duration', 'Pollutant Standard', 'Date Local', 'Units of Measure', 'Event Type', 'Observation Count', 'Observation Percent', 'Arithmetic Mean', '1st Max Value', '1st Max Hour', 'AQI', 'Method Code', 'Method Name', 'Local Site Name', 'Address', 'State Name', 'County Name', 'City Name', 'CBSA Name', 'Date of Last Change']

Sample records:


,0,1,2,3,4
State Code,01,01,01,01,01
County Code,003,003,003,003,003
Site Num,0010,0010,0010,0010,0010
Parameter Code,88101,88101,88101,88101,88101
POC,1,1,1,1,1
Latitude,30.497478,30.497478,30.497478,30.497478,30.497478
Longitude,-87.880258,-87.880258,-87.880258,-87.880258,-87.880258
Datum,NAD83,NAD83,NAD83,NAD83,NAD83
Parameter Name,PM2.5 - Local Conditions,PM2.5 - Local Conditions,PM2.5 - Local Conditions,PM2.5 - Local Conditions,PM2.5 - Local Conditions
Sample Duration,24 HOUR,24 HOUR,24 HOUR,24 HOUR,24 HOUR


In [23]:


years = [2018, 2019, 2020]
aqs_ia1_list = []

for year in years:
    zip_path = raw_dir / f"daily_88101_{year}.zip"

    with zipfile.ZipFile(zip_path, "r") as archive:
        csv_name = f"daily_88101_{year}.csv"

        with archive.open(csv_name) as csv_file:
            yearly_df = pd.read_csv(
                csv_file,
                dtype=str,
                low_memory=False
            )

    ia1_year = yearly_df[
        (yearly_df["State Code"] == "19") &
        (yearly_df["County Code"] == "113") &
        (yearly_df["Site Num"] == "0040") &
        (yearly_df["Parameter Code"] == "88101")
    ].copy()

    aqs_ia1_list.append(ia1_year)

    print(f"{year}: {len(ia1_year)} IA1 records")

aqs_ia1 = pd.concat(aqs_ia1_list, ignore_index=True)

print("\nCombined shape:", aqs_ia1.shape)
print("\nPOC counts:")
print(aqs_ia1["POC"].value_counts().sort_index())

display(
    aqs_ia1[
        [
            "Date Local",
            "POC",
            "Arithmetic Mean",
            "Observation Count",
            "Observation Percent",
            "Method Code",
            "Method Name"
        ]
    ].sort_values(["Date Local", "POC"])
)

2018: 1768 IA1 records
2019: 2104 IA1 records
2020: 2460 IA1 records

Combined shape: (6332, 29)

POC counts:
POC
1     1056
2      180
23     953
3     2102
4     2041
Name: count, dtype: int64


,Date Local,POC,Arithmetic Mean,Observation Count,Observation Percent,Method Code,Method Name
408,2018-01-01,3,2.363636,22,92.0,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
751,2018-01-01,3,2.3,1,100.0,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
1082,2018-01-01,4,5.958333,24,100.0,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
1430,2018-01-01,4,5.9,1,100.0,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
348,2018-01-02,2,12.4,1,100.0,145,R & P Model 2025 PM-2.5 Sequential Air Sampler w/VSCC - Gravimetric
...,...,...,...,...,...,...,...
4216,2020-12-31,1,10.2,1,100.0,145,R & P Model 2025 PM-2.5 Sequential Air Sampler w/VSCC - Gravimetric
5978,2020-12-31,23,9.579167,24,100.0,736,Teledyne T640 at 5.0 LPM (Corrected) - Broadband spectroscopy
6331,2020-12-31,23,9.5,1,100.0,736,Teledyne T640 at 5.0 LPM (Corrected) - Broadband spectroscopy
4631,2020-12-31,3,11.325,24,100.0,236,Teledyne T640 at 5.0 LPM - Broadband spectroscopy


In [24]:
# Inspect the fields that may distinguish multiple records
# for the same site, date, and POC.

print("Sample durations:")
print(aqs_ia1["Sample Duration"].value_counts(dropna=False))

print("\nPollutant standards:")
print(aqs_ia1["Pollutant Standard"].value_counts(dropna=False))

print("\nEvent types:")
print(aqs_ia1["Event Type"].value_counts(dropna=False))

print("\nRecords for January 1, 2018:")
display(
    aqs_ia1[
        aqs_ia1["Date Local"].eq("2018-01-01")
    ][
        [
            "Date Local",
            "POC",
            "Sample Duration",
            "Pollutant Standard",
            "Event Type",
            "Observation Count",
            "Observation Percent",
            "Arithmetic Mean",
            "Method Code",
            "Method Name"
        ]
    ].sort_values(["POC", "Pollutant Standard"])
)

Sample durations:
Sample Duration
1 HOUR           2573
24-HR BLK AVG    2523
24 HOUR          1236
Name: count, dtype: int64

Pollutant standards:
Pollutant Standard
PM25 24-hour 2012    3759
NaN                  2573
Name: count, dtype: int64

Event types:
Event Type
Included    4143
NaN         2189
Name: count, dtype: int64

Records for January 1, 2018:


,Date Local,POC,Sample Duration,Pollutant Standard,Event Type,Observation Count,Observation Percent,Arithmetic Mean,Method Code,Method Name
751,2018-01-01,3,24-HR BLK AVG,PM25 24-hour 2012,Included,1,100.0,2.3,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
408,2018-01-01,3,1 HOUR,NaN,Included,22,92.0,2.363636,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
1430,2018-01-01,4,24-HR BLK AVG,PM25 24-hour 2012,Included,1,100.0,5.9,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
1082,2018-01-01,4,1 HOUR,NaN,Included,24,100.0,5.958333,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation


In [25]:
print("Sample duration counts:")
print(aqs_ia1["Sample Duration"].value_counts(dropna=False))

print("\nPollutant standard counts:")
print(aqs_ia1["Pollutant Standard"].value_counts(dropna=False))

# Dataset A: IA1, May 13, 2018, FEM
print("\nDataset A reference records:")
display(
    df_24hr[
        (df_24hr["ID"] == "IA1") &
        (df_24hr["Date"].astype(str) == "2018-05-13") &
        (df_24hr["FEMorFRM"] == "FEM")
    ][["Date", "PM25FM", "FEMorFRMtype", "PM25cf1"]]
)

# AQS records for the same date
print("\nAQS records:")
display(
    aqs_ia1[
        aqs_ia1["Date Local"] == "2018-05-13"
    ][
        [
            "Date Local",
            "POC",
            "Sample Duration",
            "Pollutant Standard",
            "Observation Count",
            "Observation Percent",
            "Arithmetic Mean",
            "Method Code",
            "Method Name"
        ]
    ].sort_values(["POC", "Sample Duration"])
)

Sample duration counts:
Sample Duration
1 HOUR           2573
24-HR BLK AVG    2523
24 HOUR          1236
Name: count, dtype: int64

Pollutant standard counts:
Pollutant Standard
PM25 24-hour 2012    3759
NaN                  2573
Name: count, dtype: int64

Dataset A reference records:


,Date,PM25FM,FEMorFRMtype,PM25cf1
6941,2018-05-13,13.750000,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation,33.047216
7319,2018-05-13,11.958333,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation,33.047216



AQS records:


,Date Local,POC,Sample Duration,Pollutant Standard,Observation Count,Observation Percent,Arithmetic Mean,Method Code,Method Name
121,2018-05-13,1,24 HOUR,PM25 24-hour 2012,1,100.0,16.8,145,R & P Model 2025 PM-2.5 Sequential Air Sampler w/VSCC - Gravimetric
527,2018-05-13,3,1 HOUR,NaN,24,100.0,13.75,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
867,2018-05-13,3,24-HR BLK AVG,PM25 24-hour 2012,1,100.0,13.7,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
1206,2018-05-13,4,1 HOUR,NaN,24,100.0,11.958333,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
1552,2018-05-13,4,24-HR BLK AVG,PM25 24-hour 2012,1,100.0,11.9,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation


In [26]:
# Filter Dataset A to repeated IA1 FEM date groups
ia1_fem_dates = ia1_repeated_summary[
    ia1_repeated_summary["FEMorFRM"] == "FEM"
][["Date"]].copy()

# Keep only AQS hourly daily means
aqs_ia1_fem_hourly = aqs_ia1[
    (aqs_ia1["Sample Duration"] == "1 HOUR") &
    (aqs_ia1["Method Code"] == "170")
].copy()

# Match dates
comparison = ia1_fem_dates.merge(
    aqs_ia1_fem_hourly[
        ["Date Local", "POC", "Arithmetic Mean"]
    ],
    left_on="Date",
    right_on="Date Local",
    how="left"
)

print("Repeated IA1 FEM dates:", len(ia1_fem_dates))
print("Matched AQS records:", comparison["POC"].notna().sum())

display(
    comparison[
        ["Date", "POC", "Arithmetic Mean"]
    ].sort_values(["Date", "POC"])
)

Repeated IA1 FEM dates: 100
Matched AQS records: 183


,Date,POC,Arithmetic Mean
0,2018-05-12,3,7.666667
1,2018-05-12,4,5.083333
2,2018-05-13,3,13.75
3,2018-05-13,4,11.958333
4,2018-05-26,3,5.166667
...,...,...,...
178,2019-12-08,4,15.25
179,2019-12-13,4,27.833333
180,2019-12-20,4,14.875
181,2019-12-24,4,16.791667


In [28]:
# Give every original Dataset A row a unique identifier
ia1_fem_rows = ia1_fem_rows.copy()
ia1_fem_rows["dataset_row_id"] = range(len(ia1_fem_rows))

value_check = ia1_fem_rows.merge(
    aqs_compare,
    left_on="Date",
    right_on="Date Local",
    how="left"
)

value_check["absolute_difference"] = (
    value_check["PM25FM"] - value_check["Arithmetic Mean"]
).abs()

value_check["exact_match"] = (
    value_check["absolute_difference"] < 0.000001
)

# Summarize matches for each original Dataset A row
row_match_summary = (
    value_check
    .groupby("dataset_row_id")
    .agg(
        Date=("Date", "first"),
        dataset_reference=("PM25FM", "first"),
        aqs_records=("POC", "count"),
        exact_matches=("exact_match", "sum"),
        matched_pocs=(
            "POC",
            lambda x: sorted(x.dropna().unique())
        )
    )
    .reset_index()
)

print("Original Dataset A rows:", len(row_match_summary))
print(
    "Rows with exact AQS match:",
    (row_match_summary["exact_matches"] > 0).sum()
)
print(
    "Rows without exact AQS match:",
    (row_match_summary["exact_matches"] == 0).sum()
)

print("\nRows without exact matches:")
display(
    row_match_summary[
        row_match_summary["exact_matches"] == 0
    ]
)

Original Dataset A rows: 200
Rows with exact AQS match: 183
Rows without exact AQS match: 17

Rows without exact matches:


,dataset_row_id,Date,dataset_reference,aqs_records,exact_matches,matched_pocs
0,0,2019-10-18,7.754167,1,0,[4]
13,13,2019-09-12,8.279167,1,0,[4]
18,18,2019-12-20,13.983333,1,0,[4]
27,27,2019-12-08,16.945833,1,0,[4]
29,29,2019-12-24,17.683333,1,0,[4]
30,30,2019-10-01,7.145833,1,0,[4]
31,31,2019-09-28,7.316667,1,0,[4]
42,42,2019-11-15,19.862500,1,0,[4]
53,53,2019-10-19,9.725000,1,0,[4]
68,68,2019-09-20,15.258333,1,0,[4]


In [30]:
# Select all required AQS fields
aqs_compare = aqs_ia1_fem_hourly[
    [
        "Date Local",
        "POC",
        "Arithmetic Mean",
        "Method Code",
        "Method Name",
        "Sample Duration",
        "Observation Count",
        "Observation Percent"
    ]
].copy()

aqs_compare["Arithmetic Mean"] = pd.to_numeric(
    aqs_compare["Arithmetic Mean"], errors="coerce"
)

# Rebuild comparison
value_check = ia1_fem_rows.merge(
    aqs_compare,
    left_on="Date",
    right_on="Date Local",
    how="left"
)

value_check["absolute_difference"] = (
    value_check["PM25FM"] - value_check["Arithmetic Mean"]
).abs()

value_check["exact_match"] = (
    value_check["absolute_difference"] < 0.000001
)

# Identify unmatched original Dataset A rows
unmatched_ids = row_match_summary.loc[
    row_match_summary["exact_matches"] == 0,
    "dataset_row_id"
]

unmatched_comparison = value_check[
    value_check["dataset_row_id"].isin(unmatched_ids)
].copy()

unmatched_comparison["signed_difference"] = (
    unmatched_comparison["PM25FM"]
    - unmatched_comparison["Arithmetic Mean"]
)

display(
    unmatched_comparison[
        [
            "Date",
            "PM25FM",
            "POC",
            "Arithmetic Mean",
            "signed_difference",
            "Method Code",
            "Method Name",
            "Sample Duration",
            "Observation Count",
            "Observation Percent"
        ]
    ].sort_values("Date")
)

print("\nDifference summary:")
display(unmatched_comparison["signed_difference"].describe())

,Date,PM25FM,POC,Arithmetic Mean,signed_difference,Method Code,Method Name,Sample Duration,Observation Count,Observation Percent
238,2019-09-01,10.225000,4,10.583333,-0.358333,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation,1 HOUR,24,100.0
23,2019-09-12,8.279167,4,8.916667,-0.637500,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation,1 HOUR,24,100.0
119,2019-09-20,15.258333,4,14.750000,0.508333,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation,1 HOUR,24,100.0
53,2019-09-28,7.316667,4,7.333333,-0.016666,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation,1 HOUR,24,100.0
52,2019-10-01,7.145833,4,6.521739,0.624094,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation,1 HOUR,23,96.0
0,2019-10-18,7.754167,4,6.708333,1.045834,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation,1 HOUR,24,100.0
91,2019-10-19,9.725000,4,9.583333,0.141667,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation,1 HOUR,24,100.0
168,2019-10-26,17.275000,4,14.750000,2.525000,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation,1 HOUR,24,100.0
337,2019-10-27,16.241667,4,13.166667,3.075000,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation,1 HOUR,24,100.0
72,2019-11-15,19.862500,4,17.416667,2.445833,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation,1 HOUR,24,100.0



Difference summary:


count    17.000000
mean      0.877643
std       1.280761
min      -0.891667
25%      -0.016666
50%       0.624094
75%       1.695833
max       3.104167
Name: signed_difference, dtype: float64

In [31]:
# Retrieve all AQS records for the 17 unmatched dates,
# including every sample duration and POC.

unmatched_dates = unmatched_comparison["Date"].unique()

aqs_unmatched_dates = aqs_ia1[
    aqs_ia1["Date Local"].isin(unmatched_dates)
].copy()

aqs_unmatched_dates["Arithmetic Mean"] = pd.to_numeric(
    aqs_unmatched_dates["Arithmetic Mean"],
    errors="coerce"
)

display(
    aqs_unmatched_dates[
        [
            "Date Local",
            "POC",
            "Sample Duration",
            "Arithmetic Mean",
            "Observation Count",
            "Observation Percent",
            "Method Code",
            "Method Name"
        ]
    ].sort_values(["Date Local", "POC", "Sample Duration"])
)

,Date Local,POC,Sample Duration,Arithmetic Mean,Observation Count,Observation Percent,Method Code,Method Name
2009,2019-09-01,1,24 HOUR,9.700000,1,100.0,145,R & P Model 2025 PM-2.5 Sequential Air Sampler w/VSCC - Gravimetric
3628,2019-09-01,23,1 HOUR,8.991667,24,100.0,736,Teledyne T640 at 5.0 LPM (Corrected) - Broadband spectroscopy
3750,2019-09-01,23,24-HR BLK AVG,8.900000,1,100.0,736,Teledyne T640 at 5.0 LPM (Corrected) - Broadband spectroscopy
2432,2019-09-01,3,1 HOUR,10.225000,24,100.0,236,Teledyne T640 at 5.0 LPM - Broadband spectroscopy
2789,2019-09-01,3,24-HR BLK AVG,10.200000,1,100.0,236,Teledyne T640 at 5.0 LPM - Broadband spectroscopy
...,...,...,...,...,...,...,...,...
5987,2020-01-10,23,24-HR BLK AVG,12.500000,1,100.0,736,Teledyne T640 at 5.0 LPM (Corrected) - Broadband spectroscopy
4285,2020-01-10,3,1 HOUR,14.479167,24,100.0,236,Teledyne T640 at 5.0 LPM - Broadband spectroscopy
4640,2020-01-10,3,24-HR BLK AVG,14.400000,1,100.0,236,Teledyne T640 at 5.0 LPM - Broadband spectroscopy
4992,2020-01-10,4,1 HOUR,11.375000,24,100.0,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation


In [32]:
# All AQS FEM hourly daily means for IA1
aqs_all_fem_hourly = aqs_ia1[
    (aqs_ia1["Sample Duration"] == "1 HOUR") &
    (aqs_ia1["Method Code"].isin(["170", "236", "736"]))
].copy()

aqs_all_fem_hourly["Arithmetic Mean"] = pd.to_numeric(
    aqs_all_fem_hourly["Arithmetic Mean"],
    errors="coerce"
)

# Compare every Dataset A FEM row against all AQS FEM POCs
all_poc_comparison = ia1_fem_rows.merge(
    aqs_all_fem_hourly[
        ["Date Local", "POC", "Arithmetic Mean", "Method Code", "Method Name"]
    ],
    left_on="Date",
    right_on="Date Local",
    how="left"
)

all_poc_comparison["absolute_difference"] = (
    all_poc_comparison["PM25FM"]
    - all_poc_comparison["Arithmetic Mean"]
).abs()

all_poc_comparison["exact_match"] = (
    all_poc_comparison["absolute_difference"] < 0.000001
)

# Summarize by original Dataset A row
all_poc_match_summary = (
    all_poc_comparison
    .groupby("dataset_row_id")
    .agg(
        Date=("Date", "first"),
        dataset_reference=("PM25FM", "first"),
        aqs_records=("POC", "count"),
        exact_matches=("exact_match", "sum"),
        matching_pocs=(
            "POC",
            lambda x: sorted(
                all_poc_comparison.loc[
                    x.index[
                        all_poc_comparison.loc[x.index, "exact_match"]
                    ],
                    "POC"
                ].unique()
            )
        )
    )
    .reset_index()
)

print("Dataset A FEM rows:", len(all_poc_match_summary))
print(
    "Rows with exact AQS match:",
    (all_poc_match_summary["exact_matches"] > 0).sum()
)
print(
    "Rows without exact AQS match:",
    (all_poc_match_summary["exact_matches"] == 0).sum()
)

print("\nExact-match POC distribution:")
print(
    all_poc_match_summary[
        all_poc_match_summary["exact_matches"] > 0
    ]["matching_pocs"].value_counts()
)

print("\nUnmatched rows:")
display(
    all_poc_match_summary[
        all_poc_match_summary["exact_matches"] == 0
    ]
)

Dataset A FEM rows: 200
Rows with exact AQS match: 200
Rows without exact AQS match: 0

Exact-match POC distribution:
matching_pocs
[3]    100
[4]    100
Name: count, dtype: int64

Unmatched rows:


,dataset_row_id,Date,dataset_reference,aqs_records,exact_matches,matching_pocs


In [33]:
ga1_monitors = aqs_monitors[
    (aqs_monitors["State Code"] == "13") &
    (aqs_monitors["County Code"] == "089") &
    (aqs_monitors["Site Number"] == "0002") &
    (aqs_monitors["Parameter Code"] == "88101")
].copy()

print("GA1 PM2.5 monitor records:", len(ga1_monitors))

display(
    ga1_monitors[
        [
            "POC",
            "First Year of Data",
            "Last Sample Date",
            "Monitor Type",
            "Last Method Code",
            "Last Method",
            "Exclusions",
            "Reporting Agency"
        ]
    ].sort_values("POC")
)

GA1 PM2.5 monitor records: 4


,POC,First Year of Data,Last Sample Date,Monitor Type,Last Method Code,Last Method,Exclusions,Reporting Agency
87137,1,1999,2026-02-28,SLAMS,145,R & P Model 2025 PM-2.5 Sequential Air Sampler w/VSCC - Gravimetric,NaN,Georgia Air Protection Branch Ambient Monitoring Program
87138,2,2008,2026-02-28,SLAMS,145,R & P Model 2025 PM-2.5 Sequential Air Sampler w/VSCC - Gravimetric,NaN,Georgia Air Protection Branch Ambient Monitoring Program
87140,23,2017,2024-01-31,SLAMS,738,Teledyne T640X at 16.67 LPM (Corrected) - Broadband spectroscopy,NaN,Georgia Air Protection Branch Ambient Monitoring Program
87139,3,2011,2026-04-30,SLAMS,638,Teledyne T640X at 16.67 LPM w/Network Data Alignment enabled - Broadband spectroscopy,All (2017-06-14 - 2023-07-31),Georgia Air Protection Branch Ambient Monitoring Program


In [34]:
ga1_repeated = df_24hr[
    (df_24hr["ID"] == "GA1") &
    (df_24hr["FEMorFRM"].isin(["FEM", "FRM"]))
].copy()

ga1_repeated_summary = (
    ga1_repeated
    .groupby(["Date", "FEMorFRM"])
    .agg(
        rows=("PM25FM", "size"),
        reference_values=("PM25FM", lambda x: sorted(x.dropna().unique())),
        monitor_types=("FEMorFRMtype", lambda x: sorted(x.dropna().unique())),
        purpleair_cf1=("PM25cf1", "first"),
        purpleair_cfatm=("PM25cfatm", "first")
    )
    .reset_index()
)

ga1_repeated_summary = (
    ga1_repeated_summary[
        ga1_repeated_summary["rows"] > 1
    ]
    .sort_values(["Date", "FEMorFRM"])
)

print("Repeated GA1 groups:", len(ga1_repeated_summary))
print("\nRepeated groups by reference category:")
print(ga1_repeated_summary["FEMorFRM"].value_counts())

display(ga1_repeated_summary)

Repeated GA1 groups: 24

Repeated groups by reference category:
FEMorFRM
FRM    24
Name: count, dtype: int64


,Date,FEMorFRM,rows,reference_values,monitor_types,purpleair_cf1,purpleair_cfatm
7,2019-08-07,FRM,2,"[11.2, 11.3]",[R & P Model 2025 PM-2.5 Sequential Air Sampler w/VSCC - Gravimetric],22.340403,22.256868
11,2019-08-10,FRM,2,"[11.0, 11.5]",[R & P Model 2025 PM-2.5 Sequential Air Sampler w/VSCC - Gravimetric],19.211027,19.210845
17,2019-08-13,FRM,2,"[12.3, 12.5]",[R & P Model 2025 PM-2.5 Sequential Air Sampler w/VSCC - Gravimetric],25.081398,24.934896
21,2019-08-16,FRM,2,[9.5],[R & P Model 2025 PM-2.5 Sequential Air Sampler w/VSCC - Gravimetric],15.478444,15.322153
27,2019-08-19,FRM,2,"[7.7, 7.9]",[R & P Model 2025 PM-2.5 Sequential Air Sampler w/VSCC - Gravimetric],12.560181,12.560181
33,2019-08-22,FRM,2,"[9.0, 9.2]",[R & P Model 2025 PM-2.5 Sequential Air Sampler w/VSCC - Gravimetric],6.495544,6.495544
39,2019-08-25,FRM,2,"[4.8, 5.0]",[R & P Model 2025 PM-2.5 Sequential Air Sampler w/VSCC - Gravimetric],7.892799,7.892799
45,2019-08-28,FRM,2,[5.9],[R & P Model 2025 PM-2.5 Sequential Air Sampler w/VSCC - Gravimetric],9.853933,9.853933
51,2019-08-31,FRM,2,"[8.9, 9.1]",[R & P Model 2025 PM-2.5 Sequential Air Sampler w/VSCC - Gravimetric],15.917875,15.917556
57,2019-09-12,FRM,2,[14.7],[R & P Model 2025 PM-2.5 Sequential Air Sampler w/VSCC - Gravimetric],29.189653,28.039660


In [35]:
ga1_aqs_list = []

for year in [2018, 2019, 2020]:
    zip_path = raw_dir / f"daily_88101_{year}.zip"

    with zipfile.ZipFile(zip_path, "r") as archive:
        with archive.open(f"daily_88101_{year}.csv") as csv_file:
            yearly_df = pd.read_csv(
                csv_file,
                dtype=str,
                low_memory=False
            )

    ga1_year = yearly_df[
        (yearly_df["State Code"] == "13") &
        (yearly_df["County Code"] == "089") &
        (yearly_df["Site Num"] == "0002") &
        (yearly_df["Parameter Code"] == "88101")
    ].copy()

    ga1_aqs_list.append(ga1_year)
    print(f"{year}: {len(ga1_year)} GA1 records")

aqs_ga1 = pd.concat(ga1_aqs_list, ignore_index=True)

print("\nCombined GA1 shape:", aqs_ga1.shape)

print("\nPOC counts:")
print(aqs_ga1["POC"].value_counts().sort_index())

print("\nSample duration counts:")
print(aqs_ga1["Sample Duration"].value_counts(dropna=False))

print("\nMethod counts:")
print(
    aqs_ga1.groupby(["POC", "Method Code", "Method Name"])
    .size()
    .to_string()
)

2018: 1752 GA1 records
2019: 1868 GA1 records
2020: 1823 GA1 records

Combined GA1 shape: (5443, 29)

POC counts:
POC
1      914
2      241
23    2143
3     2145
Name: count, dtype: int64

Sample duration counts:
Sample Duration
1 HOUR           2155
24-HR BLK AVG    2133
24 HOUR          1155
Name: count, dtype: int64

Method counts:
POC  Method Code  Method Name                                                        
1    145          R & P Model 2025 PM-2.5 Sequential Air Sampler w/VSCC - Gravimetric     914
2    145          R & P Model 2025 PM-2.5 Sequential Air Sampler w/VSCC - Gravimetric     241
23   738          Teledyne T640X at 16.67 LPM (Corrected) - Broadband spectroscopy       2143
3    238          Teledyne T640X at 16.67 LPM - Broadband spectroscopy                   2145


In [36]:
# Dataset A: GA1 repeated FRM reference records
ga1_frm_rows = df_24hr[
    (df_24hr["ID"] == "GA1") &
    (df_24hr["FEMorFRM"] == "FRM") &
    (df_24hr["Date"].astype(str).isin(
        ga1_repeated_summary["Date"].astype(str)
    ))
].copy()

ga1_frm_rows["Date"] = ga1_frm_rows["Date"].astype(str)
ga1_frm_rows["PM25FM"] = pd.to_numeric(
    ga1_frm_rows["PM25FM"], errors="coerce"
)
ga1_frm_rows["dataset_row_id"] = range(len(ga1_frm_rows))

# AQS FRM hourly daily means
aqs_ga1_frm = aqs_ga1[
    (aqs_ga1["Sample Duration"] == "24 HOUR") &
    (aqs_ga1["Method Code"] == "145")
].copy()

aqs_ga1_frm["Arithmetic Mean"] = pd.to_numeric(
    aqs_ga1_frm["Arithmetic Mean"], errors="coerce"
)

# Match by date, retaining both FRM POCs
ga1_comparison = ga1_frm_rows.merge(
    aqs_ga1_frm[
        [
            "Date Local",
            "POC",
            "Arithmetic Mean",
            "Observation Count",
            "Observation Percent",
            "Method Code"
        ]
    ],
    left_on="Date",
    right_on="Date Local",
    how="left"
)

ga1_comparison["absolute_difference"] = (
    ga1_comparison["PM25FM"] -
    ga1_comparison["Arithmetic Mean"]
).abs()

ga1_comparison["exact_match"] = (
    ga1_comparison["absolute_difference"] < 0.000001
)

# Summarize matches per original Dataset A row
ga1_match_summary = (
    ga1_comparison
    .groupby("dataset_row_id")
    .agg(
        Date=("Date", "first"),
        dataset_reference=("PM25FM", "first"),
        aqs_records=("POC", "count"),
        exact_matches=("exact_match", "sum"),
        matching_pocs=(
            "POC",
            lambda x: sorted(
                ga1_comparison.loc[
                    x.index[
                        ga1_comparison.loc[x.index, "exact_match"]
                    ],
                    "POC"
                ].unique()
            )
        )
    )
    .reset_index()
)

print("Dataset A GA1 FRM rows:", len(ga1_match_summary))
print(
    "Rows with exact AQS match:",
    (ga1_match_summary["exact_matches"] > 0).sum()
)
print(
    "Rows without exact AQS match:",
    (ga1_match_summary["exact_matches"] == 0).sum()
)

print("\nExact-match POC distribution:")
print(
    ga1_match_summary[
        ga1_match_summary["exact_matches"] > 0
    ]["matching_pocs"].value_counts()
)

print("\nUnmatched rows:")
display(
    ga1_match_summary[
        ga1_match_summary["exact_matches"] == 0
    ]
)

Dataset A GA1 FRM rows: 48
Rows with exact AQS match: 48
Rows without exact AQS match: 0

Exact-match POC distribution:
matching_pocs
[1]       20
[2]       20
[1, 2]     8
Name: count, dtype: int64

Unmatched rows:


,dataset_row_id,Date,dataset_reference,aqs_records,exact_matches,matching_pocs


### Decision 1 — Final Conclusion

Based on the dataset investigation, published research, and targeted AQS validation, both FEM and FRM measurements will be retained as separate reference observations.

The following rules will be applied:

1. Retain FEM and FRM observations when available.
2. Do not automatically prioritize one reference category over the other.
3. Do not average FEM and FRM measurements into a single reference value.
4. Retain distinct repeated reference observations rather than automatically removing or averaging them.
5. Recognize that multiple reference observations sharing the same PurpleAir site-date measurement may be statistically dependent.
6. Report FEM and FRM performance separately where sample sizes permit.

**Limitations:** AQS validation was conducted at two sites only. The dataset does not contain a unique physical reference-monitor identifier for every observation. The final treatment of repeated observations in model evaluation will be addressed in the validation design.

**Decision Status:** Established, with limitations.

## Decision 2 — Treatment of Nonpositive Reference Measurements in Dataset A

### Analytical Question

Dataset A contains reference PM2.5 measurements (`PM25FM`) that are zero or negative.

Since PM2.5 concentrations are physically nonnegative, these observations require further investigation before deciding how they should be handled in the calibration analysis.

In this section, we will investigate:

1. How many zero and negative reference measurements are present.
2. Which PurpleAir sites, regions, dates, and reference-monitor categories contain these observations.
3. Whether the nonpositive values correspond to records available in EPA Air Quality System (AQS) data.
4. Whether the values can be explained by measurement characteristics, data-processing procedures, or potential instrument issues.
5. How retaining or excluding these observations could affect the calibration analysis.

We will examine the original dataset, corresponding AQS records, and available technical documentation before establishing the final treatment.

**The final decision will be documented after the supporting investigation and findings.**

In [40]:
# Extract nonpositive reference measurements for investigation

nonpositive_records = df_24hr.loc[
    df_24hr["PM25FM"] <= 0
].copy()

print("Total nonpositive records:", len(nonpositive_records))
print("Negative readings:", (nonpositive_records["PM25FM"] < 0).sum())
print("Zero readings:", (nonpositive_records["PM25FM"] == 0).sum())
print("Unique sensor IDs:", nonpositive_records["ID"].nunique())
print("Unique dates:", nonpositive_records["Date"].nunique())

display(
    nonpositive_records[
        ["Date", "ID", "region", "PM25FM", "FEMorFRM", "FEMorFRMtype"]
    ].sort_values(["ID", "Date"]).reset_index(drop=True)
)

Total nonpositive records: 44
Negative readings: 43
Zero readings: 1
Unique sensor IDs: 10
Unique dates: 39


,Date,ID,region,PM25FM,FEMorFRM,FEMorFRMtype
0,2018-11-24,CA10,West,-0.333333,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
1,2019-02-03,CA10,West,-0.333333,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
2,2019-02-04,CA10,West,-0.541667,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
3,2019-02-16,CA10,West,-0.166667,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
4,2018-07-29,CA15,West,-1.777778,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
5,2018-07-30,CA15,West,-1.772727,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
6,2018-07-31,CA15,West,-1.444444,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
7,2018-08-01,CA15,West,-0.083333,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
8,2018-08-08,CA15,West,-0.090909,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
9,2018-05-16,CA16,West,-1.000000,FEM,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation


In [41]:
# Identify existing EPA daily DataFrames and their relevant columns

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        required = {"State Code", "County Code", "Site Num",
                    "Date Local", "Parameter Code", "POC",
                    "Arithmetic Mean", "Method Code"}

        if required.issubset(obj.columns):
            print(f"{name}: {obj.shape}")
            print("Date range:", obj["Date Local"].min(),
                  "to", obj["Date Local"].max())
            print("Available columns:", list(obj.columns))

aqs_2018_sample: (5, 29)
Date range: 2018-01-02 to 2018-01-14
Available columns: ['State Code', 'County Code', 'Site Num', 'Parameter Code', 'POC', 'Latitude', 'Longitude', 'Datum', 'Parameter Name', 'Sample Duration', 'Pollutant Standard', 'Date Local', 'Units of Measure', 'Event Type', 'Observation Count', 'Observation Percent', 'Arithmetic Mean', '1st Max Value', '1st Max Hour', 'AQI', 'Method Code', 'Method Name', 'Local Site Name', 'Address', 'State Name', 'County Name', 'City Name', 'CBSA Name', 'Date of Last Change']
yearly_df: (704092, 29)
Date range: 2020-01-01 to 2020-12-31
Available columns: ['State Code', 'County Code', 'Site Num', 'Parameter Code', 'POC', 'Latitude', 'Longitude', 'Datum', 'Parameter Name', 'Sample Duration', 'Pollutant Standard', 'Date Local', 'Units of Measure', 'Event Type', 'Observation Count', 'Observation Percent', 'Arithmetic Mean', '1st Max Value', '1st Max Hour', 'AQI', 'Method Code', 'Method Name', 'Local Site Name', 'Address', 'State Name', 'Coun

In [42]:
import zipfile

# PurpleAir ID -> EPA AQS site
aqs_site_map = {
    "CA10": ("06", "037", "9033"),
    "CA15": ("06", "079", "2007"),
    "CA16": ("06", "079", "8002"),
    "CA17": ("06", "111", "3001"),
    "CA2":  ("06", "079", "8002"),
    "CA3":  ("06", "083", "2011"),
    "CA4":  ("06", "083", "2004"),
    "IA1":  ("19", "113", "0040"),
    "IA2":  ("19", "113", "0040"),
    "WA3":  ("53", "061", "1007")
}

# Prepare only the site/date combinations needed
match_keys = nonpositive_records[["Date", "ID"]].copy()
match_keys["AQS_site"] = match_keys["ID"].map(aqs_site_map)
match_keys["Date Local"] = pd.to_datetime(match_keys["Date"]).dt.strftime("%Y-%m-%d")

# Read the three existing EPA daily files, retaining only affected sites/dates
aqs_matches = []

for year in [2018, 2019, 2020]:
    zip_path = f"../data/raw/daily_88101_{year}.zip"

    with zipfile.ZipFile(zip_path) as z:
        csv_file = next(n for n in z.namelist() if n.lower().endswith(".csv"))

        with z.open(csv_file) as f:
            temp = pd.read_csv(f, low_memory=False, dtype={
                "State Code": str,
                "County Code": str,
                "Site Num": str,
                "POC": str,
                "Method Code": str
            })

    for sensor_id, (state, county, site) in aqs_site_map.items():
        dates = set(
            match_keys.loc[match_keys["ID"] == sensor_id, "Date Local"]
        )

        subset = temp[
            (temp["State Code"].str.zfill(2) == state) &
            (temp["County Code"].str.zfill(3) == county) &
            (temp["Site Num"].str.zfill(4) == site) &
            (temp["Parameter Code"] == 88101) &
            (temp["Date Local"].isin(dates))
        ].copy()

        if not subset.empty:
            subset["PurpleAir_ID"] = sensor_id
            aqs_matches.append(subset)

aqs_nonpositive_matches = (
    pd.concat(aqs_matches, ignore_index=True)
    if aqs_matches else pd.DataFrame()
)

# Display relevant daily measurements and quality information
quality_columns = [
    c for c in aqs_nonpositive_matches.columns
    if any(term in c.lower() for term in
           ["qualifier", "event", "duration", "observation",
            "method", "mean", "poc", "date local"])
]

print("AQS matching records:", len(aqs_nonpositive_matches))
print("Matched PurpleAir IDs:",
      sorted(aqs_nonpositive_matches["PurpleAir_ID"].unique())
      if not aqs_nonpositive_matches.empty else [])

display(
    aqs_nonpositive_matches[
        ["PurpleAir_ID"] + quality_columns
    ].sort_values(["PurpleAir_ID", "Date Local", "POC"])
)

AQS matching records: 97
Matched PurpleAir IDs: ['CA10', 'CA15', 'CA16', 'CA17', 'CA2', 'CA3', 'CA4', 'IA1', 'IA2', 'WA3']


,PurpleAir_ID,POC,Sample Duration,Date Local,Event Type,Observation Count,Observation Percent,Arithmetic Mean,Method Code,Method Name
0,CA10,1,1 HOUR,2018-11-24,NaN,24,100.0,-0.333333,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
1,CA10,1,24-HR BLK AVG,2018-11-24,NaN,1,100.0,-0.300000,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
46,CA10,1,1 HOUR,2019-02-03,NaN,24,100.0,-0.333333,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
49,CA10,1,24-HR BLK AVG,2019-02-03,NaN,1,100.0,-0.300000,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
47,CA10,1,1 HOUR,2019-02-04,NaN,24,100.0,-0.541667,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
...,...,...,...,...,...,...,...,...,...,...
90,IA2,3,24-HR BLK AVG,2019-05-23,Included,1,100.0,0.400000,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
92,IA2,4,1 HOUR,2019-05-23,Included,24,100.0,-0.583333,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
94,IA2,4,24-HR BLK AVG,2019-05-23,Included,1,100.0,-0.500000,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation
95,WA3,5,1 HOUR,2019-01-03,NaN,22,92.0,-0.454545,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation


In [44]:
# Compare Dataset A nonpositive readings with matching AQS daily records
import numpy as np

dataset_a_check = nonpositive_records[
    ["Date", "ID", "PM25FM"]
].copy().reset_index(drop=True)

dataset_a_check["record_id"] = dataset_a_check.index
dataset_a_check["Date Local"] = pd.to_datetime(
    dataset_a_check["Date"]
).dt.strftime("%Y-%m-%d")

aqs_check = aqs_nonpositive_matches[
    [
        "PurpleAir_ID", "Date Local", "POC",
        "Sample Duration", "Arithmetic Mean",
        "Method Code", "Observation Count",
        "Observation Percent", "Event Type"
    ]
].copy()

comparison_44 = dataset_a_check.merge(
    aqs_check,
    left_on=["ID", "Date Local"],
    right_on=["PurpleAir_ID", "Date Local"],
    how="left"
)

comparison_44["difference"] = (
    comparison_44["Arithmetic Mean"] - comparison_44["PM25FM"]
)

comparison_44["exact_match"] = np.isclose(
    comparison_44["Arithmetic Mean"],
    comparison_44["PM25FM"],
    atol=1e-5,
    rtol=0,
    equal_nan=False
)

# Summarize whether each Dataset A record has an exact AQS match
match_summary_44 = (
    comparison_44.groupby("record_id")
    .agg(
        ID=("ID", "first"),
        Date=("Date", "first"),
        PM25FM=("PM25FM", "first"),
        AQS_records=("Arithmetic Mean", "count"),
        exact_AQS_matches=("exact_match", "sum")
    )
    .reset_index()
)

print("Dataset A records:", len(dataset_a_check))
print(
    "Records with at least one exact AQS match:",
    (match_summary_44["exact_AQS_matches"] > 0).sum()
)
print(
    "Records with no exact AQS match:",
    (match_summary_44["exact_AQS_matches"] == 0).sum()
)

print("\nMatches by AQS sample duration:")
display(
    comparison_44.groupby("Sample Duration", dropna=False)
    .agg(
        AQS_records=("Arithmetic Mean", "count"),
        exact_matches=("exact_match", "sum")
    )
)

print("\nRecord-level results:")
display(
    match_summary_44.sort_values(["ID", "Date"])
)

print("\nDetailed AQS comparisons:")
display(
    comparison_44[
        [
            "ID", "Date", "PM25FM", "POC",
            "Sample Duration", "Arithmetic Mean",
            "difference", "exact_match",
            "Method Code", "Observation Count",
            "Observation Percent", "Event Type"
        ]
    ].sort_values(["ID", "Date", "POC", "Sample Duration"])
)

Dataset A records: 44
Records with at least one exact AQS match: 43
Records with no exact AQS match: 1

Matches by AQS sample duration:


,AQS_records,exact_matches
Sample Duration,,
1 HOUR,47,43
24 HOUR,3,0
24-HR BLK AVG,47,4



Record-level results:


,record_id,ID,Date,PM25FM,AQS_records,exact_AQS_matches
0,0,CA10,2018-11-24,-0.333333,2,1
1,1,CA10,2019-02-03,-0.333333,2,1
2,2,CA10,2019-02-04,-0.541667,2,1
3,3,CA10,2019-02-16,-0.166667,2,1
4,4,CA15,2018-07-29,-1.777778,2,1
5,5,CA15,2018-07-30,-1.772727,2,1
6,6,CA15,2018-07-31,-1.444444,2,1
7,7,CA15,2018-08-01,-0.083333,2,1
8,8,CA15,2018-08-08,-0.090909,2,1
11,11,CA16,2018-05-16,-1.000000,2,2



Detailed AQS comparisons:


,ID,Date,PM25FM,POC,Sample Duration,Arithmetic Mean,difference,exact_match,Method Code,Observation Count,Observation Percent,Event Type
0,CA10,2018-11-24,-0.333333,1,1 HOUR,-0.333333,0.000000,True,170,24,100.0,NaN
1,CA10,2018-11-24,-0.333333,1,24-HR BLK AVG,-0.300000,0.033333,False,170,1,100.0,NaN
2,CA10,2019-02-03,-0.333333,1,1 HOUR,-0.333333,0.000000,True,170,24,100.0,NaN
3,CA10,2019-02-03,-0.333333,1,24-HR BLK AVG,-0.300000,0.033333,False,170,1,100.0,NaN
4,CA10,2019-02-04,-0.541667,1,1 HOUR,-0.541667,0.000000,True,170,24,100.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
87,IA2,2019-05-23,-0.583333,3,24-HR BLK AVG,0.400000,0.983333,False,170,1,100.0,Included
88,IA2,2019-05-23,-0.583333,4,1 HOUR,-0.583333,0.000000,True,170,24,100.0,Included
89,IA2,2019-05-23,-0.583333,4,24-HR BLK AVG,-0.500000,0.083333,False,170,1,100.0,Included
95,WA3,2019-01-03,-0.454545,5,1 HOUR,-0.454545,0.000000,True,170,22,92.0,NaN


In [45]:
# Investigate the one unmatched Dataset A record: CA2, 2018-01-09

display(
    comparison_44.loc[
        (comparison_44["ID"] == "CA2") &
        (comparison_44["Date Local"] == "2018-01-09")
    ].sort_values(["POC", "Sample Duration"])
)

# Show all EPA daily records for the corresponding AQS site and date
display(
    aqs_nonpositive_matches.loc[
        (aqs_nonpositive_matches["PurpleAir_ID"] == "CA2") &
        (aqs_nonpositive_matches["Date Local"] == "2018-01-09")
    ].sort_values(["POC", "Sample Duration"])[
        [
            "POC", "Date Local", "Sample Duration",
            "Arithmetic Mean", "Observation Count",
            "Observation Percent", "Method Code",
            "Method Name", "Event Type"
        ]
    ]
)

,Date,ID,PM25FM,record_id,Date Local,PurpleAir_ID,POC,Sample Duration,Arithmetic Mean,Method Code,Observation Count,Observation Percent,Event Type,difference,exact_match
34,2018-01-09,CA2,-1.130435,17,2018-01-09,CA2,3,1 HOUR,-1.217391,170,23,96.0,NaN,-0.086956,False
35,2018-01-09,CA2,-1.130435,17,2018-01-09,CA2,3,24-HR BLK AVG,-1.200000,170,1,100.0,NaN,-0.069565,False


,POC,Date Local,Sample Duration,Arithmetic Mean,Observation Count,Observation Percent,Method Code,Method Name,Event Type
23,3,2018-01-09,1 HOUR,-1.217391,23,96.0,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation,NaN
35,3,2018-01-09,24-HR BLK AVG,-1.200000,1,100.0,170,Met One BAM-1020 Mass Monitor w/VSCC - Beta Attenuation,NaN


In [46]:
# Inspect AQS record history for the CA2 discrepancy

display(
    aqs_nonpositive_matches.loc[
        (aqs_nonpositive_matches["PurpleAir_ID"] == "CA2") &
        (aqs_nonpositive_matches["Date Local"] == "2018-01-09"),
        [
            "POC",
            "Date Local",
            "Sample Duration",
            "Arithmetic Mean",
            "Observation Count",
            "Observation Percent",
            "Method Code",
            "Event Type",
            "Date of Last Change"
        ]
    ]
)

,POC,Date Local,Sample Duration,Arithmetic Mean,Observation Count,Observation Percent,Method Code,Event Type,Date of Last Change
23,3,2018-01-09,1 HOUR,-1.217391,23,96.0,170,NaN,2024-05-18
35,3,2018-01-09,24-HR BLK AVG,-1.200000,1,100.0,170,NaN,2024-05-21


### Decision 2 — Final Conclusion

Based on the dataset investigation, EPA AQS comparison, and available technical documentation, the 44 nonpositive reference measurements will be preserved in the original dataset and flagged for separate consideration.

The investigation identified:

- 44 nonpositive reference measurements: 43 negative values and one zero.
- All 44 observations are FEM measurements from Met One BAM-1020 instruments, using method code 170.
- The observations occur across 10 PurpleAir IDs and two regions.
- 43 of the 44 observations exactly match the corresponding EPA AQS daily 1-hour arithmetic means.
- One observation (CA2, 2018-01-09) does not exactly match the corresponding AQS value. The available records do not establish the reason for this discrepancy.

### Selected Approach

The following rules will guide the analysis:

1. Preserve all 44 original nonpositive reference measurements.
2. Do not automatically delete, replace, or modify these values.
3. Flag nonpositive reference measurements so their influence can be examined separately.
4. Include a sensitivity analysis that repeats the relevant evaluation after excluding all 44 observations.
5. Do not use percentage-error metrics such as MAPE when reference measurements are zero or negative.
6. Any logarithmic transformation must account for nonpositive values separately and must not silently discard them.

### Limitations

- Matching an observation against an EPA AQS record confirms agreement with the reported AQS value; it does not independently establish that the measurement was physically valid.
- The available evidence does not establish the actual cause of the negative measurements.
- Historical instrument maintenance records, zero tests, calibration records, and operator investigations were unavailable.
- The CA2 discrepancy remains unresolved.

### Decision Status

**Established, with limitations.**

The original observations will be retained and flagged. Their influence will be evaluated through sensitivity analysis.

The final choice of observations used in each model evaluation will be documented alongside the results.

### Decision 3: Handling missing meteorological variables

**Decision:** Preserve missing values in `RH`, `TempC`, and `Dewpoint`. Do not apply arbitrary mean, median, or constant-value imputation during the initial data preparation stage.

**1. Missingness findings**

Dataset A contains the following missing values:

| Variable | Missing observations | Missing percentage |
|---|---:|---:|
| RH | 17 | 0.14% |
| TempC | 184 | 1.48% |
| Dewpoint | 184 | 1.48% |

Additional observations:

- All 17 observations with missing RH also have missing temperature and dew point.
- The other 167 observations have missing temperature and dew point while RH remains available.
- Of the 184 observations with missing temperature, 170 are in the North region, 11 in the West, 2 in Central Southwest, and 1 in the Southeast.
- The missingness is particularly concentrated in sensor WI4: 167 of its 185 observations (90.27%) have missing temperature, mainly during May–October 2019.

**2. Decision rationale**

1. Missing values should not be replaced without a defensible methodological reason.
2. The concentration of missingness by region, sensor, and period suggests that missingness may not be random.
3. Arbitrary imputation could introduce artificial relationships or distort the influence of meteorological variables.
4. Raw sensor-to-reference comparisons do not require temperature or dew point, so these missing values should not unnecessarily reduce the baseline evaluation dataset.

**3. Analysis handling**

- Preserve the original missing values.
- Do not remove observations from the entire dataset solely because a meteorological variable is missing.
- For models using RH, use observations with available RH values.
- For models using temperature or dew point, apply the appropriate feature-availability rule and document the resulting sample size.
- Compare calibration approaches on consistent eligible evaluation observations wherever possible.
- If imputation is considered later, select and justify the method using training data only, without using held-out evaluation data.
- Report sample sizes and missingness patterns for each model comparison.

**4. Limitations**

The reason for the missing meteorological measurements has not been established. The concentration of missing temperature values in the North region and WI4 sensor may affect regional comparisons and model generalization.

**Status:** Provisional decision. Preserve missing values and evaluate feature-specific eligibility during model development.

## Decision 4 — Selection of the Raw PurpleAir PM2.5 Measurement

### 4.1 Analytical Question

Dataset A contains two raw PurpleAir PM2.5 measurements:

- `PM25cf1`: AB-channel averaged higher CF (CF=1).
- `PM25cfatm`: AB-channel averaged lower CF (CF=ATM).

The dataset also contains previously corrected measurements.

The analytical questions are:

1. Which raw PurpleAir measurement should be used as the primary input for calibration?
2. Does the choice between CF=1 and CF=ATM affect calibration performance?
3. How should existing corrected measurements be treated when developing independent calibration models?

### 4.2 Dataset Documentation

The Dataset A data dictionary defines:

- `PM25cf1`: AB-channel averaged higher CF PM2.5 concentration (µg/m³).
- `PM25cfatm`: AB-channel averaged lower CF PM2.5 concentration (µg/m³).
- `PM25PAlinearcor`: PurpleAir PM2.5 corrected using a linear correction developed over the full dataset.
- `PM25PARHcor`: PurpleAir PM2.5 corrected using a relative-humidity correction developed over the full dataset.

Dataset B's `PA` field also represents raw PurpleAir CF=1 PM2.5.

### 4.3 Supporting Research

Barkjohn et al. (2021) evaluated both CF=1 and CF=ATM measurements as candidate inputs for calibration models.

The study reported lower model error using CF=1 than CF=ATM across the model forms evaluated. The difference was modest in that dataset, partly because only 3.8% of observations had reference PM2.5 concentrations above 20 µg/m³, where the two measurement fields exhibit more different relationships.

The study's selected humidity-aware correction used CF=1 as its PM2.5 input.

### 4.4 Initial Analytical Approach

Based on the available documentation and research, the initial approach is:

1. Use `PM25cf1` as the primary raw PurpleAir measurement candidate.
2. Retain `PM25cfatm` for secondary comparison and sensitivity analysis.
3. Use `PM25FM` as the reference outcome, subject to the reference-selection and nonpositive-measurement decisions.
4. Do not use `PM25PAlinearcor` or `PM25PARHcor` as raw predictors in independently trained calibration models.
5. Compare CF=1 and CF=ATM using consistent eligible observations and evaluation splits.

### 4.5 Limitations and Open Questions

- The associated study's finding that CF=1 produced lower error does not guarantee that it will perform better in every region or concentration range.
- The relationship may differ under wildfire-smoke conditions.
- Our own evaluation must establish whether the same performance pattern holds in the available datasets.
- The final model input specification may be revised based on EDA and subsequent evaluation.

### 4.6 Decision Status

**Provisional approach:** CF=1 is the primary raw measurement candidate, while CF=ATM is retained for sensitivity analysis.

The final input specification will be confirmed through exploratory analysis and model evaluation.

## Decision 5 — Selection of Meteorological Predictors for Calibration

### 5.1 Analytical Question

Dataset A contains three meteorological variables that may help explain differences between PurpleAir measurements and reference PM2.5 measurements:

- Relative humidity (`RH`)
- Temperature (`TempC`)
- Dew point (`Dewpoint`)

The analytical questions are:

1. Does relative humidity improve calibration performance?
2. Does temperature provide additional predictive information?
3. Does dew point provide useful information beyond relative humidity and temperature?
4. How will missing meteorological observations affect model comparisons?

### 5.2 Dataset Documentation

The Dataset A data dictionary defines:

- `RH`: Relative humidity (%) measured by PurpleAir.
- `TempC`: Temperature (°C) measured by PurpleAir.
- `Dewpoint`: Dew point (°C) calculated from PurpleAir temperature and relative humidity.

Dew point is therefore a derived variable rather than an independent measurement.

### 5.3 Initial Model Candidates

The following model specifications will be considered initially:

| Candidate | Predictors | Analytical purpose |
|---|---|---|
| Baseline | `PM25cf1` | Establish raw sensor performance |
| Model 1 | `PM25cf1`, `RH` | Investigate humidity-aware calibration |
| Model 2 | `PM25cf1`, `RH`, `TempC` | Investigate the additional contribution of temperature |

Dew point will be retained for exploratory analysis and may be considered in alternative model specifications.

### 5.4 Decision Rationale

1. Relative humidity is a relevant environmental factor when evaluating optical particle-sensor measurements.
2. The associated EPA research provides a basis for evaluating humidity-aware PurpleAir corrections.
3. Temperature may provide additional information, but its contribution must be evaluated empirically.
4. Dew point is calculated from RH and temperature. Including all three simultaneously in an initial model may introduce redundant information and complicate interpretation.
5. Comparing progressively expanded model candidates can help evaluate the contribution of additional predictors.

### 5.5 Missing-Data Considerations

The data profile identified:

- 17 missing RH observations.
- 184 missing TempC observations.
- 184 missing Dewpoint observations.
- All 17 missing RH observations also have missing TempC and Dewpoint.
- Temperature missingness is concentrated in the North region, particularly at WI4.

Consequently, models requiring temperature may represent a different regional and temporal sample from the baseline and RH-only models.

Initial handling principles:

- Preserve missing meteorological values during EDA.
- Do not automatically impute missing values.
- Report the eligible sample size for each model.
- Compare candidate models on identical eligible evaluation observations where possible.
- Fit any required imputation procedure using training data only.

### 5.6 Limitations and Open Questions

- The contribution of temperature and dew point has not yet been established.
- Missingness may affect regional representation and model comparability.
- The final meteorological predictor combination should depend on observed relationships and subsequent model evaluation.
- Other model specifications may be considered if EDA reveals relevant patterns.

### 5.7 Decision Status

**Provisional approach:** RH is the primary meteorological predictor candidate, temperature is an additional candidate, and dew point is retained for exploratory analysis.

The final predictor set and model specifications will be determined after EDA and empirical evaluation.

## Decision 6 — Repeated Observations and Prevention of Data Leakage

### 6.1 Analytical Question

Dataset A contains multiple observations associated with the same PurpleAir site and date. Some groups contain separate FEM and FRM measurements or multiple reference readings within the same category.

These observations may share the same underlying PurpleAir measurement and therefore may not be statistically independent.

The analytical questions are:

1. How should repeated observations be handled during model development?
2. How can data leakage be prevented when dividing observations into training and testing datasets?
3. What validation strategy is appropriate for evaluating generalization to previously unseen PurpleAir sites?
4. How should the evaluation distinguish performance on known sites from performance on unseen sites?

### 6.2 Findings from the Data Profile

- Dataset A contains 12,430 observations and 9,707 unique `ID`–`Date`–`Region` combinations.
- There are 2,199 combinations with repeated observations.
- Of the 1,146 combinations containing both FEM and FRM measurements, 665 have exactly one FEM and one FRM observation, while 481 have repeated measurements on at least one side.
- Repeated observations can represent distinct AQS POCs and must not automatically be treated as duplicate records.
- Multiple reference observations can share the same PurpleAir site-date measurement.
- The `ID` field represents a study PurpleAir site identifier, not necessarily a unique physical sensor serial number.

### 6.3 Initial Data-Leakage Prevention Principles

The following principles will guide the evaluation design:

1. Individual rows sharing the same underlying PurpleAir site-date measurement should not be randomly divided between training and testing datasets.
2. All observations sharing the same `ID` and `Date` should remain in the same data partition.
3. Repeated FEM/FRM and POC-level observations will remain separate, consistent with Decision 1.
4. Potential statistical dependence between observations sharing the same PurpleAir site-date measurement must be considered when interpreting evaluation results.
5. A grouped split based on `ID` is a candidate strategy for evaluating generalization to previously unseen PurpleAir sites.

### 6.4 Evaluation Questions

Two different evaluation objectives should be distinguished:

- **Held-out observations:** Evaluate predictions for observations from sites that may also be represented in the training data.
- **Held-out sites:** Evaluate whether a model generalizes to PurpleAir sites that were not represented in training.

Holding out sites does not automatically establish generalization to entirely unseen geographic regions. Regional evaluation would require a separate strategy.

### 6.5 Limitations and Open Questions

- Dataset A contains 49 PurpleAir IDs across five regions.
- The distribution of observations across these IDs may affect the feasibility of grouped cross-validation.
- Some regions may have limited numbers of sites or observations.
- The appropriate number of folds and validation method cannot be established without examining the site and regional distributions.

### 6.6 Decision Status

**Established principle:** Prevent leakage by keeping observations sharing the same underlying PurpleAir site-date measurement in the same partition.

**Pending EDA:** The exact training/testing split, grouping strategy, cross-validation method, and number of folds will be selected after examining the distribution of observations across IDs and regions.

## Decision 7 — Model Evaluation Metrics

### 7.1 Analytical Question

The project aims to determine whether calibration improves agreement between low-cost PurpleAir PM2.5 measurements and reference-monitor measurements.

The analytical questions are:

1. Which metrics best describe prediction error and systematic bias?
2. How should model performance be compared consistently?
3. How should performance differences across reference categories, regions, and concentration ranges be reported?

### 7.2 Initial Evaluation Metrics

The following metrics will be considered:

1. **Mean Absolute Error (MAE):** Measures the average absolute difference between predicted and reference PM2.5 concentrations.
2. **Root Mean Squared Error (RMSE):** Measures the square root of the average squared prediction error and gives greater weight to larger errors.
3. **Mean Bias Error (MBE):** Measures systematic overestimation or underestimation. It will be defined as prediction minus reference, so positive values indicate overestimation and negative values indicate underestimation.
4. **Coefficient of Determination (R²):** Describes the proportion of variation in reference measurements explained by the predictions under the evaluation definition used.

Where appropriate, regression slope and intercept will also be considered to examine systematic differences between PurpleAir predictions and reference measurements.

### 7.3 Initial Reporting Principles

- Report performance for the uncalibrated baseline and calibrated model candidates.
- Compare models using identical eligible evaluation observations.
- Report the number of observations used alongside each metric.
- Report FEM and FRM results separately where sample sizes permit.
- Examine regional and PM2.5 concentration-range differences where sufficient observations are available.
- Do not rely on R² alone to determine calibration accuracy.
- Avoid percentage-error metrics such as MAPE when reference measurements are zero or negative.
- Examine the effect of excluding the 44 nonpositive reference observations through the sensitivity analysis described in Decision 2.

### 7.4 Limitations and Open Questions

- Metric interpretation may be affected by repeated observations and statistical dependence.
- The appropriate concentration ranges and regional reporting groups will be informed by EDA.
- Comparisons must account for differences in eligible sample sizes.
- The final reporting structure may be refined after examining the data distributions and evaluation objectives.

### 7.5 Decision Status

**Initial metric framework established:** MAE, RMSE, MBE, and R² will be considered as the primary evaluation metrics.

The final reporting groups, comparisons, and any additional metrics will be confirmed after EDA and evaluation-design review.

## Decision 8 — Calibration Model Comparison

### 8.1 Analytical Question

The central objective of the project is to determine how much calibration improves agreement between low-cost PurpleAir measurements and reference-monitor PM2.5 measurements.

The analytical questions are:

1. How does the uncalibrated PurpleAir measurement compare with the reference measurement?
2. Does a fitted calibration relationship improve prediction accuracy?
3. Does incorporating meteorological information improve calibration?
4. Are observed improvements consistent across sites, regions, and PM2.5 concentration ranges?

### 8.2 Initial Model Candidates

Based on the data dictionary and supporting research, the following comparisons will be considered:

1. **Uncalibrated baseline:** Raw `PM25cf1` compared directly with reference PM2.5.
2. **Linear calibration:** A linear relationship between raw `PM25cf1` and reference PM2.5.
3. **Humidity-aware calibration:** A model incorporating `RH` alongside raw `PM25cf1`.
4. **Extended meteorological model:** A model evaluating whether adding `TempC` provides additional improvement beyond the humidity-aware candidate.

`PM25cfatm` will remain available for a secondary sensitivity comparison.

Existing corrected fields (`PM25PAlinearcor` and `PM25PARHcor`) will not be used as raw predictors in independently fitted calibration models.

### 8.3 Initial Evaluation Principles

- Fit independently developed calibration models using training data only.
- Apply fitted models to held-out observations without refitting.
- Compare calibrated predictions against the uncalibrated baseline using the same eligible evaluation observations.
- Consider MAE, RMSE, MBE, and R² as evaluation metrics.
- Examine whether performance improvements remain consistent across held-out sites and regions where appropriate.
- Report sample sizes and the evaluation strategy alongside results.

### 8.4 Limitations and Open Questions

- The initial model candidates are not an exhaustive list of possible calibration approaches.
- EDA may reveal nonlinear relationships, concentration-dependent errors, or interactions that justify additional model specifications.
- The contribution of meteorological predictors must be established empirically.
- The final model selection must consider generalization performance, not only training performance.
- The exact validation strategy remains dependent on the findings and methodological considerations in Decision 6.

### 8.5 Decision Status

**Initial comparison framework established.**

The uncalibrated baseline and initial calibration candidates will guide further analysis. The final model specifications, additional candidate models, and model-selection criteria will be determined after EDA and validation-design review.

## Decision 9 — Generalization Under Wildfire-Smoke Conditions

### 9.1 Analytical Question

An important objective of this project is to determine whether calibration improvements observed under general measurement conditions also hold under wildfire-smoke conditions.

The analytical questions are:

1. Does calibration improve agreement during wildfire-smoke conditions?
2. Does calibration performance change at elevated PM2.5 concentrations?
3. Do different adjustment approaches produce different levels of agreement?
4. How do the available withholding approaches affect the interpretation of generalization performance?
5. Can findings from Dataset A be meaningfully compared with results from Dataset B?

### 9.2 Dataset Roles

The two datasets have different temporal resolutions and analytical structures.

**Dataset A — 24-Hour Collocation**

Contains daily PurpleAir and reference-monitor measurements. It will be used to investigate daily calibration relationships and their performance across the available sites and regions.

**Dataset B — Hourly Wildfire Smoke**

Contains hourly PurpleAir and reference measurements, relative humidity, and previously published adjusted measurements under different adjustment and withholding configurations.

It will be used to investigate calibration behavior under the wildfire-smoke conditions represented in the dataset.

### 9.3 Initial Analytical Considerations

The following questions will guide subsequent investigation:

1. Compare raw and adjusted PurpleAir measurements against reference measurements under wildfire-smoke conditions.
2. Examine whether performance changes across PM2.5 concentration ranges.
3. Investigate the supplied adjustment approaches (A–F) and withholding approaches (LOBD and LOSO).
4. Account for repeated configuration versions of the same underlying city-sensor-hour observation.
5. Avoid treating repeated configuration rows as independent underlying observations.
6. Consider the differences in temporal resolution, sampling structure, and evaluation protocols when interpreting results from Dataset A and Dataset B.

### 9.4 Limitations and Open Questions

- Dataset A and Dataset B are not directly interchangeable because they have different temporal resolutions and sampling structures.
- Dataset B contains repeated adjustment and withholding configurations for the same underlying observations.
- The primary Dataset B adjustment and withholding benchmark has not yet been selected.
- It remains to be established whether Dataset A calibration models can be independently evaluated using Dataset B or whether the supplied adjustment configurations should be analyzed as a separate comparison.
- The exact Dataset B evaluation procedure requires review of the source documentation and analytical structure.

### 9.5 Decision Status

**Research objective established; evaluation design pending.**

The project will investigate whether calibration improvements hold under wildfire-smoke conditions. The specific Dataset B benchmark, adjustment comparison, withholding strategy, and relationship between the two datasets will not be finalized until the relevant documentation and data structure have been examined.

## Decision 10 — Data Preprocessing and Transformation Rules

### 10.1 Analytical Question

Before calibration modeling, the datasets must be prepared in a consistent and reproducible manner.

The analytical questions are:

1. How should missing values be handled?
2. How should extreme or unusual PM2.5 measurements be investigated?
3. When are transformations or scaling necessary?
4. How can preprocessing be performed without introducing data leakage?
5. How can the original source data and analytical changes be preserved?

### 10.2 Initial Preprocessing Principles

**1. Missing values**

- Preserve missing values during initial analysis.
- Do not replace missing values with arbitrary constants or averages.
- Investigate missingness patterns and their potential effect on regional and temporal representation.
- Any imputation used in modeling must be fitted using training data only.
- Report the number of observations available to each model.

**2. Outliers and unusual values**

- Do not automatically remove extreme PM2.5 measurements.
- Investigate unusual values using the source data and relevant context.
- Preserve original observations and document any exclusions.
- Evaluate the impact of justified exclusions through sensitivity analysis.

**3. Nonpositive reference measurements**

- Preserve and flag the 44 nonpositive reference observations identified in Decision 2.
- Evaluate the impact of excluding these observations through a separate sensitivity analysis.
- Do not use percentage-error metrics when reference measurements are zero or negative.
- Any transformation requiring strictly positive values must explicitly account for nonpositive observations.

**4. Feature preparation**

- Use `PM25cf1` as the primary raw PurpleAir measurement candidate, subject to Decision 4.
- Retain `PM25cfatm` for secondary sensitivity comparisons.
- Do not use existing corrected PM2.5 fields as raw predictors in independently fitted calibration models.
- Consider meteorological variables according to the candidate specifications in Decision 5.

**5. Transformations and scaling**

- Do not apply transformations or scaling without an analytical reason.
- Investigate variable distributions and relationships before selecting transformations.
- If scaling or transformation is required, estimate its parameters using training data only.
- Apply the same fitted transformation to corresponding test data.

**6. Reproducibility**

- Preserve the original raw datasets.
- Perform preprocessing in reproducible notebook steps.
- Document filtering, exclusions, imputation, transformations, and changes to analytical datasets.
- Keep derived variables and analytical subsets separate from the original source data.

### 10.3 Limitations and Open Questions

- Specific outlier thresholds cannot be established before examining the distributions.
- The need for transformations or scaling will depend on the observed data and selected models.
- Imputation decisions will depend on missingness patterns and model requirements.
- Any preprocessing choices that affect the evaluation sample must be documented and justified.

### 10.4 Decision Status

**Initial preprocessing principles established.**

Specific outlier handling, transformations, scaling, and imputation procedures will be determined after EDA and before model fitting.